# Preparação de Dados: Transações Imobiliárias de Fortaleza (ITBI)

**Objetivo:** Este notebook tem como finalidade realizar a limpeza, seleção de features e transformação de coordenadas do dataset público de ITBI de Fortaleza. O arquivo resultante será utilizado para alimentar um Dashboard interativo (desenvolvido em Dash/Python) focado na análise espacial do mercado imobiliário.

**Etapas:**
1. Importação das bibliotecas e carregamento dos dados.
2. Seleção de colunas, filtros e padronização para otimização de memória.
3. Transformação de Coordenadas Geográficas (SIRGAS 2000 UTM para WGS84 Lat/Lon).
4. Criação de novas variáveis (Valor do Metro Quadrado).
5. Exportação do dataset limpo.

In [1]:
# Importando as bibliotecas necessárias
import pandas as pd
from pyproj import Transformer
import warnings
from pathlib import Path

# Suprimindo avisos para manter o notebook limpo
warnings.filterwarnings('ignore')

## 1. Carregamento dos Dados
Os dados originais foram obtidos no portal Dados Abertos de Fortaleza.

In [ ]:
# Carregar o dataset original em Excel
caminho_arquivo = Path.cwd().resolve().parent / "dataset" / "dados_abertos_itbi_transacoes_imobiliarias.xlsx"
df = pd.read_excel(caminho_arquivo)

print(f"Dimensões do dataset original: {df.shape}")

Dimensões do dataset original: (94970, 31)


## 2. Seleção, Filtros e Padronização
Selecionamos apenas as colunas úteis para o Dashboard, filtramos os imóveis relevantes e padronizamos os nomes das colunas para letras minúsculas.

In [ ]:
# Seleção de Colunas
colunas_selecionadas = [
    'EXERCICIO', 'BAIRRO', 'LOGRADOURO', 'XSIRGAS2000', 'YSIRGAS2000', 
    'AREA_EDIFICADA', 'TIPO_USO_IMOVEL', 'PADRAO_CONSTRUCAO',
    'VL_BASE_CALCULO', 'DATA_DA_TRANSACAO_ITBI'
]
df_dash = df[colunas_selecionadas].copy()

# Filtro de Qualidade: Apenas Residencial e Comercial
df_dash = df_dash[df_dash["TIPO_USO_IMOVEL"].isin(["Residencial", "Comercial"])]

# Padronização de Colunas: Transformar tudo em letras minúsculas
df_dash.columns = df_dash.columns.str.lower()

print("Novas colunas padronizadas:")
print(list(df_dash.columns))

## 3. Limpeza e Transformação de Coordenadas Geográficas
Remoção de registros sem localização e conversão do sistema SIRGAS 2000 (UTM 24S) para o padrão de web maps WGS84 (Latitude/Longitude).

In [ ]:
# Tratamento de Valores Nulos nas Coordenadas
df_dash = df_dash.dropna(subset=['xsirgas2000', 'ysirgas2000'])
print(f"Dimensões após filtro de uso e limpeza de nulos: {df_dash.shape}")

# Transformação de Coordenadas
transformer = Transformer.from_crs("EPSG:31984", "EPSG:4326", always_xy=True)

# Aplicando a conversão usando as colunas em minúsculo
lon, lat = transformer.transform(df_dash['xsirgas2000'].values, df_dash['ysirgas2000'].values)

# Adicionar as novas colunas
df_dash['longitude'] = lon
df_dash['latitude'] = lat

# Descartar as colunas SIRGAS originais para poupar espaço
df_dash = df_dash.drop(columns=['xsirgas2000', 'ysirgas2000'])

## 4. Engenharia de Variáveis e Datas
Para garantir a integridade da análise de preços, filtramos propriedades com `area_edificada` inferior a 15m². Valores menores que esse geralmente representam erros de preenchimento, terrenos sem construção averbada ou venda de frações ideais (como vagas de garagem), que distorceriam o cálculo do Metro Quadrado.

In [ ]:
# Filtro de Qualidade Física: Manter apenas imóveis com área maior que 15m²
# Isso limpa o dataset e já resolve naturalmente o problema matemático de divisão por zero
df_dash = df_dash[df_dash['area_edificada'] >= 15]

# Criar a nova métrica de preço (agora com dados limpos)
df_dash['valor_m2'] = df_dash['vl_base_calculo'] / df_dash['area_edificada']

# Tratamento de Data: Garante o formato datetime e extrai apenas a data (sem a hora)
df_dash['data_da_transacao_itbi'] = pd.to_datetime(df_dash['data_da_transacao_itbi'], errors='coerce').dt.date

# Visualizar o resultado final e a nova quantidade de linhas
print(f"Dimensões após filtro de área: {df_dash.shape}")
display(df_dash.head())

## 5. Exportação
Salvando o dataset final e otimizado que será consumido pela aplicação Dash.

In [ ]:
# Exportar o dataset refinado
nome_arquivo_saida = 'dataset_filtered.xlsx'
df_dash.to_excel(nome_arquivo_saida, index=False)

print(f"Sucesso! Dataset exportado e pronto para o Dash: {nome_arquivo_saida}")